In [ ]:
import bloodline as bl
import pandas as pd

@bl.update_data_lineage(
    default_source=bl.Source(
        source_type=bl.get_source_type("DATA_SOURCE"),
        source_metadata={"name": "retail_users"}
    )
)
def load_retail_users():
    """

    TODO: what if the user calls read_csv? Should the DATA_SOURCE be set automatically? What about
    the table name? Maybe Bloodline should be more opinionated.

    """
    return pd.DataFrame([
        {
            "user_id": 1,
            "name": "John Doe",
            "age": 30
        },
        {
            "user_id": 2,
            "name": "Jane Smith",
            "age": 25
        }
    ])

@bl.update_data_lineage(
    default_source=bl.Source(
        source_type=bl.get_source_type("DATA_SOURCE"),
        source_metadata={"name": "web_users"}
    )
)
def load_web_users():
    return pd.DataFrame([
        {
            "user_id": 3,
            "name": "Alice Johnson",
            "age": 28
        },
        {
            "user_id": 4,
            "name": "Bob Brown",
            "age": 35
        }
    ])

@bl.update_data_lineage(
    default_source=bl.Source(
        source_type=bl.get_source_type("DATA_SOURCE"),
        source_metadata={"name": "items"}
    )
)
def load_items():
    return pd.DataFrame([
        {
            "item_id": 1,
            "product": "Laptop",
            "price": 1200
        },
        {
            "item_id": 2,
            "product": "Smartphone",
            "price": 800
        }
    ])


@bl.update_data_lineage(
    default_source=bl.Source(
        source_type=bl.get_source_type("DATA_SOURCE"),
        source_metadata={"name": "purchases"}
    )
)
def load_purchases():
    return pd.DataFrame([
        {
            "id": 1,
            "user_id": 1,
            "item_id": 2,
            "quantity": 1
        },
        {
            "id": 2,
            "user_id": 3,
            "item_id": 1,
            "quantity": 2
        }
    ])


users = pd.concat([load_retail_users(), load_web_users()])
items = load_items()
purchases = load_purchases()

@bl.update_data_lineage(
    inheritance={
        "id_purchase": "",
    }
)
def make_one_big_table():
    """

    TODO: having to call the .lineage accessor explicitly doesn't make the user fall into the pit
    of success. User shouldn't have to update their code to use Bloodline out of the box.

    """
    merged = purchases.lineage.merge(users, on="user_id")
    merged = merged.lineage.merge(items, on="item_id")
    return merged

one_big_table = make_one_big_table()
one_big_table

,id,user_id,item_id,quantity,name,age,product,price,data_lineage
0,1,1,2,1,John Doe,30,Smartphone,800,"{'id': {'source_type': 'DATA_SOURCE', 'source_..."
1,2,3,1,2,Alice Johnson,28,Laptop,1200,"{'id': {'source_type': 'DATA_SOURCE', 'source_..."


In [ ]:
"""

TODO: having a global RELATIONSHIPS variable is a bit clunky. What if we instantiated a configurable
Bloodline object that held the relationships and other metadata? This object could also provide the
decorator and replace bl.update_data_lineage.

"""

bl.RELATIONSHIPS

{Relationship(left_name='purchases', left_key=('item_id',), right_name='items', right_key=('item_id',), relationship_type=<RelationshipType.ONE_TO_ONE: 'ONE_TO_ONE'>),
 Relationship(left_name='purchases', left_key=('user_id',), right_name='retail_users', right_key=('user_id',), relationship_type=<RelationshipType.ONE_TO_ONE: 'ONE_TO_ONE'>),
 Relationship(left_name='purchases', left_key=('user_id',), right_name='web_users', right_key=('user_id',), relationship_type=<RelationshipType.ONE_TO_ONE: 'ONE_TO_ONE'>)}

In [2]:
import pprint

pprint.pprint(one_big_table["data_lineage"].iloc[0])

{'age': {'source_metadata': {'name': 'retail_users'},
         'source_type': 'DATA_SOURCE'},
 'id': {'source_metadata': {'name': 'purchases'}, 'source_type': 'DATA_SOURCE'},
 'item_id': {'source_metadata': {'name': 'items'},
             'source_type': 'DATA_SOURCE'},
 'name': {'source_metadata': {'name': 'retail_users'},
          'source_type': 'DATA_SOURCE'},
 'price': {'source_metadata': {'name': 'items'}, 'source_type': 'DATA_SOURCE'},
 'product': {'source_metadata': {'name': 'items'},
             'source_type': 'DATA_SOURCE'},
 'quantity': {'source_metadata': {'name': 'purchases'},
              'source_type': 'DATA_SOURCE'},
 'user_id': {'source_metadata': {'name': 'retail_users'},
             'source_type': 'DATA_SOURCE'}}


In [3]:
pprint.pprint(one_big_table["data_lineage"].iloc[-1])

{'age': {'source_metadata': {'name': 'web_users'},
         'source_type': 'DATA_SOURCE'},
 'id': {'source_metadata': {'name': 'purchases'}, 'source_type': 'DATA_SOURCE'},
 'item_id': {'source_metadata': {'name': 'items'},
             'source_type': 'DATA_SOURCE'},
 'name': {'source_metadata': {'name': 'web_users'},
          'source_type': 'DATA_SOURCE'},
 'price': {'source_metadata': {'name': 'items'}, 'source_type': 'DATA_SOURCE'},
 'product': {'source_metadata': {'name': 'items'},
             'source_type': 'DATA_SOURCE'},
 'quantity': {'source_metadata': {'name': 'purchases'},
              'source_type': 'DATA_SOURCE'},
 'user_id': {'source_metadata': {'name': 'web_users'},
             'source_type': 'DATA_SOURCE'}}
